# 跨靶点臂 PBC 校正 —— 36 条轨迹（6 分子 × 2 受体 × 3 重复）

> **本 notebook 由 `make_cross_notebooks.py` 从主臂 V3 版定点替换生成，非重写。**
> 与主臂版的全部差异即该脚本的替换表，每一处均断言命中次数。
> **白名单 `ALLOWED_ARMS` 已刻意加入 `cross_solv_v3`** —— 原白名单会正确地拒绝本臂；
> 扩充它是 2026-10-19 的一次决定，须随本 notebook 一同入档。
> 主臂产物（`step7_solv_v3` / `step6e_solv_v3` / `*_SOLV_V3.csv`）不受本notebook 影响。

---

# phase2 · PBC 校正（跨靶点臂 cross_solv_v3）

把每条溶剂化轨迹 `06_md/md.xtc` 做周期性边界校正 → `06_md/md_fit.xtc`，
供 phase3（gmx_MMPBSA）与 phase4（MDAnalysis）使用。

**两趟 trjconv**：① `-pbc mol -center -ur compact`（整分子 + 居中蛋白）→ ② `-fit rot+trans`（骨架最小二乘对齐）。

**相对 V2 的改动**
- `ARMS = ['cross_solv_v3']`；V2 的安全断言 `endswith('_solv')` 会**拒绝** `cross_solv_v3`，
  改为显式白名单（该断言在 V2 中出现两次，两处都已改）。
- `pbc_correct()` 内的路径断言改为按 `ARMS` 判定。
- 目标数 120 → **36**（跨靶点臂：6 分子 × 2 受体 × 3 重复；主臂为 99）。
- 组名 `Protein` / `Backbone` / `System` 在 V3 index 中同名存在，**无须改动**。

**安全**：只新增 `md_fit.xtc`，绝不改动 `md.xtc`；绝不触及真空 `md/`、`md_cross/`
与 V2 溶剂化臂 `md_solv/`、`md_cross_solv/`。

In [7]:
# === 配置 ===
import os, glob, time, random, subprocess, json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

ROOT  = "/mnt/hpc/home/wanghongmei/lang2"
GMXRC = "/mnt/hpc/home/wanghongmei/apps/gromacs-2025.4-cuda12.1/bin/GMXRC"
ARMS  = ["cross_solv_v3"]                  # 只跑跨靶点臂（3V3M + 4YLU）
N_EXPECT = 36   # 依据 AMD-2 2026-10-31：该臂以两批达到 36（30 + 6）

CENTER_GROUP = "Protein"     # 第 1 趟居中组
FIT_GROUP    = "Backbone"    # 第 2 趟拟合组
OUTPUT_GROUP = "System"      # 输出原子组（gmx_MMPBSA 自行剥水）

N_PARALLEL = 6
OVERWRITE  = False
DRYRUN     = False

# 安全断言（替代 V2 的 endswith('_solv')，后者会拒绝 md_solv_v3）
ALLOWED_ARMS = {"md_solv", "md_cross_solv", "md_solv_v3", "md_cross_solv_v3",
                "cross_solv_v3"}        # ← 跨靶点臂目录，2026-10-19 刻意加入
assert set(ARMS) <= ALLOWED_ARMS, f"ARMS 含未授权目录: {set(ARMS) - ALLOWED_ARMS}"
assert not (set(ARMS) & {"md", "md_cross"}), "拒绝写入真空目录"
print("配置 OK | ARMS =", ARMS, "| 真空 md/ md_cross/ 与 V2 溶剂化臂不受影响")

配置 OK | ARMS = ['cross_solv_v3'] | 真空 md/ md_cross/ 与 V2 溶剂化臂不受影响


In [8]:
def gmx(args, stdin_text=None, cwd=None, timeout=10800):
    """在 sourced GMXRC 环境下执行 gmx；set +u 包掉 GMXRC 的 unbound-variable 告警。"""
    cmd = f'set +u; source "{GMXRC}" >/dev/null 2>&1; set -u; ' + \
          " ".join(f'"{a}"' if " " in a else a for a in args)
    p = subprocess.run(["bash", "-c", cmd], input=stdin_text,
                       capture_output=True, text=True, cwd=cwd, timeout=timeout)
    return p.returncode, p.stdout, p.stderr

rc, o, e = gmx(["gmx", "--version"])
print("gmx 可用" if rc == 0 else "⚠ gmx 不可用, 检查 GMXRC")
print((o + e).splitlines()[0] if (o or e) else "")

gmx 可用
                         :-) GROMACS - gmx, 2025.4 (-:


In [9]:
def list_targets():
    todo, exist, notready = [], [], []
    for arm in ARMS:
        for gro in sorted(glob.glob(f"{ROOT}/{arm}/*/*/06_md/md.gro")):
            d = os.path.dirname(gro)
            if "_validate" in d:
                continue
            if not all(os.path.exists(f"{d}/{f}") for f in ("md.tpr", "md.xtc", "index.ndx")):
                notready.append(d); continue
            if os.path.exists(d + "/md_fit.xtc") and not OVERWRITE:
                exist.append(d); continue
            todo.append(d)
    return todo, exist, notready

todo, exist, notready = list_targets()
print(f"待校正: {len(todo)}   已存在 md_fit: {len(exist)}   缺文件跳过: {len(notready)}")
print(f"(目标总数 = 待校正 + 已存在 = {len(todo)+len(exist)}; 全部完成应 = {N_EXPECT})")
assert set(ARMS) <= ALLOWED_ARMS

待校正: 6   已存在 md_fit: 30   缺文件跳过: 0
(目标总数 = 待校正 + 已存在 = 36; 全部完成应 = 36)


In [10]:
def pbc_correct(d):
    tpr, xtc, ndx = d + "/md.tpr", d + "/md.xtc", d + "/index.ndx"
    tmp, out = d + "/_pbc_whole.xtc", d + "/md_fit.xtc"
    assert any(f"/{a}/" in out for a in ARMS), f"拒绝: 输出不在授权臂内 -> {out}"
    rc, o, e = gmx(["gmx", "trjconv", "-s", tpr, "-f", xtc, "-n", ndx,
                    "-o", tmp, "-pbc", "mol", "-center", "-ur", "compact"],
                   stdin_text=f"{CENTER_GROUP}\n{OUTPUT_GROUP}\n")
    if rc != 0 or not os.path.exists(tmp):
        return (d, "FAIL", "pass1: " + (e.strip().splitlines()[-1] if e.strip() else "fail"))
    rc, o, e = gmx(["gmx", "trjconv", "-s", tpr, "-f", tmp, "-n", ndx,
                    "-o", out, "-fit", "rot+trans"],
                   stdin_text=f"{FIT_GROUP}\n{OUTPUT_GROUP}\n")
    try: os.remove(tmp)
    except OSError: pass
    if rc != 0 or not os.path.exists(out):
        return (d, "FAIL", "pass2: " + (e.strip().splitlines()[-1] if e.strip() else "fail"))
    return (d, "OK", f"{os.path.getsize(out)/1e9:.2f} GB")

print("pbc_correct() 已定义")

pbc_correct() 已定义


In [11]:
# === 执行 ===
if DRYRUN:
    for d in todo: print("  [计划]", d.replace(ROOT + "/", ""))
    print(f"\nDRYRUN: {len(todo)} 条待校正(未执行)。置 DRYRUN=False 再跑。")
else:
    t0, ok, fail = time.time(), 0, []
    with ThreadPoolExecutor(max_workers=N_PARALLEL) as ex:
        futs = {ex.submit(pbc_correct, d): d for d in todo}
        for i, fut in enumerate(as_completed(futs), 1):
            src = futs[fut]
            try:
                d, st, msg = fut.result()
            except Exception as exc:                      # 超时 / IO 错误：记录并继续
                d, st, msg = src, "ERR", f"{type(exc).__name__}: {exc}"[:80]
            print(f"[{i}/{len(todo)}] {st:4s} {d.replace(ROOT+'/','')}  {msg}", flush=True)
            if st == "OK": ok += 1
            else: fail.append((d, msg))
    print(f"\n完成 {ok}/{len(todo)}   用时 {(time.time()-t0)/60:.1f} min")
    for d, m in fail: print("  FAIL", d.replace(ROOT + "/", ""), "|", m)
    if fail:
        print(f"\n{len(fail)} 条未完成。重跑本 notebook 即可 —— 已完成的会被 exist 跳过。")
    # 清理可能残留的中间文件（仅删本流程自建的 _pbc_whole.xtc）
    leftovers = glob.glob(f"{ROOT}/{ARMS[0]}/*/*/06_md/_pbc_whole.xtc")
    for p in leftovers: os.remove(p)
    if leftovers: print(f"已清理 {len(leftovers)} 个残留的 _pbc_whole.xtc")

[1/6] OK   cross_solv_v3/pool_mol_001__4YLU__primary/r3/06_md  3.28 GB
[2/6] OK   cross_solv_v3/pool_mol_001__4YLU__primary/r2/06_md  3.28 GB
[3/6] OK   cross_solv_v3/pool_mol_001__4YLU__primary/r1/06_md  3.28 GB
[4/6] OK   cross_solv_v3/pool_mol_001__3V3M__primary/r1/06_md  3.53 GB
[5/6] OK   cross_solv_v3/pool_mol_001__3V3M__primary/r3/06_md  3.53 GB
[6/6] OK   cross_solv_v3/pool_mol_001__3V3M__primary/r2/06_md  3.53 GB

完成 6/6   用时 3.2 min


In [12]:
# === 验证：帧数 + md.xtc 未被改动 + 组名探测 ===
# === 公共辅助：逐轨迹按名解析 index 组（V2=UNL / V3=MOL，不写死）===
import re, glob
def ndx_groups(ndx):
    return [m.group(1) for m in
            (re.match(r'\s*\[\s*(.+?)\s*\]', ln) for ln in open(ndx)) if m]

def lig_group(ndx, candidates=("MOL", "UNL")):
    """返回 (配体组名, 组号)。命中不唯一即报错，不猜。"""
    g = ndx_groups(ndx)
    hit = [c for c in candidates if c in g]
    if len(hit) != 1:
        raise AssertionError(f"{ndx}: 配体组不唯一或缺失，命中 {hit}；现有组 {g}")
    return hit[0], g.index(hit[0])

def rec_group(ndx, name="Protein"):
    g = ndx_groups(ndx)
    assert g.count(name) == 1, f"{ndx}: '{name}' 组不唯一"
    return g.index(name)

pool = (todo or exist)
for d in random.sample(pool, min(3, len(pool))):
    rc, o, e = gmx(["gmx", "check", "-f", d + "/md_fit.xtc"])
    print(d.replace(ROOT + "/", ""))
    for l in [l.strip() for l in (o+e).splitlines() if ("Coords" in l or "Last frame" in l)][:2]:
        print("   ", l)
    print("    md.xtc 存在:", os.path.exists(d + "/md.xtc"),
          "| 配体组:", lig_group(d + "/index.ndx"))

n = sum(len(glob.glob(f"{ROOT}/{a}/*/*/06_md/md_fit.xtc")) for a in ARMS)
print(f"\n全部 md_fit 数: {n} (目标 {N_EXPECT})")
print("注: 本流程只新增 md_fit.xtc, 不改动 md.xtc。")

cross_solv_v3/pool_mol_001__3V3M__primary/r1/06_md
    Last frame      10000 time 100000.000
    Coords       10001    10
    md.xtc 存在: True | 配体组: ('MOL', 13)
cross_solv_v3/pool_mol_001__4YLU__primary/r1/06_md
    Last frame      10000 time 100000.000
    Coords       10001    10
    md.xtc 存在: True | 配体组: ('MOL', 13)
cross_solv_v3/pool_mol_001__3V3M__primary/r2/06_md
    Last frame      10000 time 100000.000
    Coords       10001    10
    md.xtc 存在: True | 配体组: ('MOL', 13)

全部 md_fit 数: 36 (目标 36)
注: 本流程只新增 md_fit.xtc, 不改动 md.xtc。


## 下一步
1. `md_fit.xtc` 出齐 36 → phase3（MM-PBSA）与 phase4（结构分析）。
2. 两者的配体组名均由 `index.ndx` 逐轨迹探测（V3 = `MOL`），不写死。

## 与 V2 的差异一览
| | V2 | V3 |
|---|---|---|
| `ARMS` | `md_solv`, `md_cross_solv` | `cross_solv_v3` |
| 安全断言 | `endswith('_solv')` | 显式白名单（原断言会拒绝 `cross_solv_v3`） |
| 目标数 | 120 | 36（跨靶点臂；主臂 99） |
| 组名 | Protein / Backbone / System | 同（无须改动） |